# FinanceGPT - Fine Tuning Notebook

This notebook is used for:
- Loading trainable language models
- Preparing LoRA fine-tuning
- Loading processed datasets
- Understanding AI training workflow
- Preparing FinanceGPT for custom learning

## Installing Required Libraries

In [1]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
!pip install transformers==4.40.2 -q

!pip install datasets -q

!pip install peft==0.11.1 -q

!pip install trl==0.8.6 -q

!pip install accelerate==0.30.1 bitsandbytes -q

## Importing Required Libraries

In [3]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments
)

from datasets import (
    load_dataset,
    load_from_disk
)

from peft import (
    LoraConfig,
    get_peft_model
)

from trl import SFTTrainer

## Understanding Fine Tuning

Fine-tuning means adapting a pretrained AI model
to perform better on a custom dataset.

Instead of training from scratch,
we improve an existing model using finance-specific data.

## Loading Base Model

In [4]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [5]:
tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
model = AutoModelForCausalLM.from_pretrained(
    model_name
)

## Understanding LoRA

LoRA (Low Rank Adaptation) allows efficient fine-tuning
by training only small adapter layers instead of
updating the entire model.

## Configuring LoRA

In [7]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [8]:
model = get_peft_model(
    model,
    lora_config
)

In [9]:
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


## Loading Processed Dataset

In [10]:
dataset = load_dataset(
    "gbharti/finance-alpaca",
    split="train[:500]"
)

In [11]:
print(dataset)

Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 500
})


## Configuring Training Arguments

In [12]:
training_args = TrainingArguments(
    output_dir="./financegpt-output",
    per_device_train_batch_size=1,
    max_steps=20,
    logging_steps=10,
    save_steps=50,
    learning_rate=2e-4,
    fp16=True,
    report_to="none"
)

## Preparing Trainer

In [13]:
def format_prompt(example):
    return f"""
### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}
"""

dataset = dataset.map(
    lambda x: {"text": format_prompt(x)}
)

In [14]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=1024,
    args=training_args
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
max_steps is given, it will override any value given in num_train_epochs


## Starting Fine Tuning

The model will now begin learning
from the finance-specific dataset.

In [15]:
trainer.train()

Step,Training Loss
10,2.456200
20,2.551900


TrainOutput(global_step=20, training_loss=2.5040638923645018, metrics={'train_runtime': 7.419, 'train_samples_per_second': 2.696, 'train_steps_per_second': 2.696, 'total_flos': 42695244509184.0, 'train_loss': 2.5040638923645018, 'epoch': 0.04})

## Saving Fine Tuned Model

In [16]:
model.save_pretrained(
    "financegpt-finetuned"
)

tokenizer.save_pretrained(
    "financegpt-finetuned"
)

('financegpt-finetuned/tokenizer_config.json',
 'financegpt-finetuned/special_tokens_map.json',
 'financegpt-finetuned/tokenizer.model',
 'financegpt-finetuned/added_tokens.json',
 'financegpt-finetuned/tokenizer.json')

## Final Observation

FinanceGPT has now completed:
- dataset preparation
- tokenization
- LoRA setup
- fine-tuning preparation
- custom finance training

This marks the beginning of FinanceGPT
becoming a domain-specific AI assistant.